# zeroshot

> Typed answers without a trained head: a two-tower model's similarity between the state and each option, or a masked language model's own prediction at an answer slot.

In [ ]:
#| default_exp zeroshot

In [ ]:
#| export
import json

import numpy as np
import torch
from fastcore.basics import store_attr

from sysone.core import *
from sysone.losses import fit_temperatures, fit_thresholds

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
from pathlib import Path
import hashlib, logging
from sysone.evaluate import evaluate
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

A sysone head learns to score anchors, and before it has seen labelled data it knows nothing. Some models can answer a typed question without any head, and such a zero-shot answer is worth having for two reasons: it is the baseline a trained head has to beat, and it is all a new task has before anyone labels anything. Two kinds of model can do it, and this module turns each into a backend shaped like a `Decider`: `predict` and `predict_batch` return answers in the Jev schema, `calibrate` fits temperatures on held-out records, and `evaluate` scores it like any other backend.

```{mermaid}
flowchart LR
    S["state, question, options"] --> T{"which model?"}
    T -- "two towers<br/>(ProtST, CLIP, a sentence encoder)" --> A["embed the state; embed each option's text;<br/>logit = scale × cosine"]
    T -- "a masked language model" --> B["one prompt ending in a mask;<br/>logit = the model's score for each<br/>option's verbalizer token there"]
    A --> P["÷ temperature, softmax:<br/>the Jev answer"]
    B --> P
```

## Two towers: similarity

A two-tower model embeds its two inputs into one space, trained so that a state and a text that describe each other lie close together: CLIP for images and captions, ProtST for proteins and their Swiss-Prot descriptions, a sentence encoder for two texts. Each option becomes a text, its description or else its key, and its logit is the cosine between the state's embedding and the option text's, times the model's `scale` (the logit scale CLIP and ProtST learn; 1 for a plain sentence encoder). This is how ProtST and ProTrek classify proteins without training, and how CLIP classifies images.

A noul needs two texts, one per answer. With descriptions of its own (`criteria={"false": "a soluble protein", "true": "a membrane protein"}`) the two are compared directly. Without them the statement is compared with its negation, which two-tower models read poorly, since contrastive pre-training rarely sees a "not". The texts come from `option_texts`, a [generic function](00_core.ipynb#generic-functions-and-multiple-dispatch) with a method per kind of question: a choice's descriptions (or keys), a score's levels, a noul's two sides. A backend that wants other texts for one kind adds a method for it.

In [ ]:
#| export
class Embedder:
    "Items to unit vectors, one row per item: states (texts, proteins, images) or option texts"
    def embed(self, items:list) -> np.ndarray: raise NotImplementedError
    def __call__(self, items) -> np.ndarray:
        v = np.asarray(self.embed(list(items)), dtype=np.float32)
        return v / np.clip(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12, None)

class SentenceTransformerEmbedder(Embedder):
    "A sentence-transformers model (install `sysone[shortlist]`)"
    def __init__(self, model:str="BAAI/bge-small-en-v1.5", prefix:str=""): store_attr(); self._enc = None
    def embed(self, items):
        if self._enc is None:
            from sentence_transformers import SentenceTransformer
            self._enc = SentenceTransformer(self.model)
        return self._enc.encode([self.prefix + str(x) for x in items])

class CachedEmbedder(Embedder):
    "An embedder that remembers each item's vector, so deciders sharing it (one per prompt, say) embed each state once"
    def __init__(self, embedder:Embedder): self.embedder, self.memo = embedder, {}
    def embed(self, items):
        missing = list(dict.fromkeys(x for x in items if x not in self.memo))
        if missing: self.memo.update(zip(missing, self.embedder(missing)))
        return np.stack([self.memo[x] for x in items])

@dispatch
def option_texts(q:Choice, *, prompt:str="{text}", negation:str="It is not the case that {text}") -> list:
    "A choice's option texts, in key order: each label's description, or the label itself"
    return [prompt.format(text=render_criterion(v) if v not in (None, "") else str(k), key=str(k)) for k, v in q.criteria.items()]

@dispatch
def option_texts(q:Score, *, prompt:str="{text}", negation:str="It is not the case that {text}") -> list:
    "A score's option texts: its levels' descriptions"
    return [prompt.format(text=render_criterion(c), key=str(i)) for i, c in enumerate(q.criteria)]

@dispatch
def option_texts(q:Noul, *, prompt:str="{text}", negation:str="It is not the case that {text}") -> list:
    "A noul's two texts, false then true: its criteria's, else the statement's negation and the statement"
    crit = q.criteria or {}
    f = crit.get("false") or negation.format(text=q.instructions[:1].lower() + q.instructions[1:])
    t = crit.get("true") or q.instructions
    return [prompt.format(text=render_criterion(f), key="false"), prompt.format(text=render_criterion(t), key="true")]

def _parsed(s):
    "A state as stored or given: JSON text becomes the structure it encodes"
    if isinstance(s, str) and s[:1] in "{[":
        try: return json.loads(s)
        except json.JSONDecodeError: pass
    return s

def _fit_zero_shot(backend, records, min_rows:int, act_accuracy:float|None=None) -> tuple:
    "Temperatures for a zero-shot backend from its logits on labelled records, and act thresholds on its calibrated confidence"
    records = [load_record(r) for r in records]
    rows = [(i, q, g) for i, r in enumerate(records) for qid, qd in r["questions"].items()
            for q in [Question.from_dict(qd, name=qid)] for g in [(r.get("gold") or {}).get(qid)] if q.answerable(g) >= 0]
    if not rows: return {}, None
    z = backend.row_logits([records[i] for i, _, _ in rows], [q for _, q, _ in rows])
    fit = [(zi, q, q.target(g)[0]) for zi, (_, q, g) in zip(z, rows) if q.answerable(g) == 1]
    temps = {}
    if fit:
        K = max(q.k for _, q, _ in fit)
        pad = lambda v, k, c: np.pad(np.asarray(v, dtype=np.float64), (0, K - k), constant_values=c)
        L = np.stack([pad(zi, q.k, -1e4) for zi, q, _ in fit])
        T = np.stack([pad(t, q.k, 0.0) for _, q, t in fit])
        M = np.stack([np.arange(K) < q.k for _, q, _ in fit])
        temps = fit_temperatures(L, T, M, np.array([q.qtype for _, q, _ in fit]), min_rows)
    if act_accuracy is None: return temps, None
    conf, right = [], []
    for zi, (_, q, g) in zip(z, rows):                 # acting is right where the gold is an option and the top one is it
        zz = np.asarray(zi, dtype=np.float64) / lookup_temperature(temps, q.qtype, q.k); p = np.exp(zz - zz.max()); p /= p.sum()
        conf.append(p.max()); right.append(q.answerable(g) == 1 and int(p.argmax()) == q.target(g)[1])
    return temps, fit_thresholds(conf, right, [q.qtype for _, q, _ in rows], [q.k for _, q, _ in rows], act_accuracy, min_rows)

class SimilarityDecider:
    "Zero-shot typed answers from a two-tower model: `scale` times the cosine between a state's embedding and each option text's"
    def __init__(self,
                 state_embedder, # states (or their `state_key` part) to unit vectors
                 text_embedder, # option texts to unit vectors, in the same space
                 scale:float=1.0, # the model's logit scale (CLIP's, ProtST's)
                 prompt:str="{text}", # how an option becomes a text: {text} is its description or key, {key} its key
                 state_key:str|None=None, # embed state[state_key] of a dict state (a protein's sequence); the whole state otherwise
                 temperatures:dict|None=None, # per type and option-count bucket, as `calibrate` fits them
                 act_threshold:float|dict|None=None, # act where the calibrated top probability ≥ this (`calibrate(act_accuracy=...)` fits them)
                 digits:int|None=4, # rounding of reported numbers
                 batch_size:int=16, # states per embedding call
                 name:str="similarity"):
        store_attr(but="temperatures"); self.temperatures, self._texts = dict(temperatures or {}), {}

    def __repr__(self): return f"SimilarityDecider({self.name} · scale {self.scale:.3g} · prompt {self.prompt!r} · {len(self.temperatures)} temperatures)"

    def _state(self, s):
        s = _parsed(s)
        if self.state_key is not None and isinstance(s, dict): return s.get(self.state_key)
        return s if isinstance(s, str) else render_state(s)

    def state_vectors(self, states:list) -> np.ndarray:
        "The states' embeddings: each distinct state embedded once, in batches of `batch_size`"
        items = [self._state(s) for s in states]
        key = lambda x: x if isinstance(x, str) else id(x)
        uniq = list({key(x): x for x in items}.values())
        at = {key(x): i for i, x in enumerate(uniq)}
        V = np.concatenate([self.state_embedder(uniq[i:i + self.batch_size]) for i in range(0, len(uniq), self.batch_size)])
        return V[[at[key(x)] for x in items]]

    def option_vectors(self, q:Question) -> np.ndarray:
        "The embeddings of a question's option texts (each text embedded once, then remembered)"
        texts = option_texts(q, prompt=self.prompt)
        missing = list(dict.fromkeys(t for t in texts if t not in self._texts))
        if missing: self._texts |= dict(zip(missing, self.text_embedder(missing)))
        return np.stack([self._texts[t] for t in texts])

    def row_logits(self, records:list, questions:list) -> list:
        "Each (record, question) pair's option logits"
        S = self.state_vectors([r["state"] for r in records])
        return [self.scale * (self.option_vectors(q) @ s) for s, q in zip(S, questions)]

    def _answer(self, q:Question, z) -> Answer:
        z = np.asarray(z, dtype=np.float64) / lookup_temperature(self.temperatures, q.qtype, q.k)
        p = np.exp(z - z.max()); p /= p.sum()
        return fill_answer(q, p, self.digits, act=float(p.max()) if self.act_threshold is not None else None, act_threshold=self.act_threshold)

    def predict_batch(self, states:list, questions:dict) -> list:
        "Answers for many states; each state is embedded once for all its questions"
        qs, S = as_questions(questions), self.state_vectors(list(states))
        z = {qid: self.scale * S @ self.option_vectors(q).T for qid, q in qs.items()}
        return [{qid: self._answer(q, z[qid][i]) for qid, q in qs.items()} for i in range(len(S))]

    def predict(self, state, questions:dict) -> dict: return self.predict_batch([state], questions)[0]

    def calibrate(self, records:list, min_rows:int=10, act_accuracy:float|None=None) -> dict:
        "Fit temperatures per question type and option-count bucket on labelled records (the model is unchanged); `act_accuracy` also fits act thresholds"
        self.temperatures, thr = _fit_zero_shot(self, records, min_rows, act_accuracy)
        if thr is not None: self.act_threshold = thr
        return self.temperatures

The tests use a stand-in two-tower model: character trigrams hashed into 64 dimensions, the same function for both towers, so a text is close to the texts it shares trigrams with:

In [ ]:
class HashEmbedder(Embedder):
    "Character trigrams hashed into 64 dimensions: a stand-in two-tower model"
    def embed(self, items):
        out = np.zeros((len(items), 64), np.float32)
        for i, s in enumerate(items):
            s = f"  {str(s).lower()}  "
            for j in range(len(s) - 2): out[i, int(hashlib.md5(s[j:j + 3].encode()).hexdigest(), 16) % 64] += 1
        return out

he = HashEmbedder()
zs = SimilarityDecider(he, he, scale=10)
qs = {"dept": {"type": "choice", "instructions": "Which team?", "criteria": {"billing": "invoices and refunds", "tech": "bugs and crashes"}},
      "urgent": {"type": "noul", "instructions": "It is urgent.", "criteria": {"false": "it can wait", "true": "urgent, today"}}}
ans = zs.predict("the app crashes with a bug on start, fix it today, urgent", qs)
test_eq((ans["dept"]["choice"], ans["urgent"]["noul"] > 0.5), ("tech", True))
test_eq(option_texts(Question.from_dict({"type": "noul", "instructions": "The photo shows a hot dog."})),
        ["It is not the case that the photo shows a hot dog.", "The photo shows a hot dog."])
test_eq(zs.predict_batch(["refund my invoice", "it crashes"], qs)[0]["dept"]["choice"], "billing")
ans

{'dept': {'type': 'choice',
  'choice': 'tech',
  'probabilities': {'billing': 0.1959, 'tech': 0.8041},
  'confidence': 0.2864,
  'answer_confidence': 0.8041},
 'urgent': {'type': 'noul',
  'noul': 0.8952,
  'confidence': 0.8952,
  'answer_confidence': 0.8952}}

A state held as a dict can hand one part to the state tower (`state_key`), the way a protein's sequence goes to the protein tower. `calibrate` fits temperatures on labelled records exactly as a learner does, and `evaluate` scores the result like any other backend:

In [ ]:
qd = {"dept": qs["dept"]}
recs = [{"state": {"sequence": s, "note": "n/a"}, "questions": qd, "gold": {"dept": {"label": g}}}
        for s, g in [("invoice refund please", "billing"), ("crash bug on login", "tech"), ("refunds for invoices", "billing"),
                     ("bugs and crashes today", "tech")] * 4]
zk = SimilarityDecider(he, he, scale=10, state_key="sequence")
temps = zk.calibrate(recs, min_rows=4)
test_eq(set(temps) >= {"choice", "choice:2"}, True)

A zero-shot decider has no act head, but it can still say when to escalate. `calibrate(act_accuracy=...)` also fits act thresholds on its calibrated top probability, per question type and option-count bucket, and its answers then carry `action`:

In [ ]:
za = SimilarityDecider(he, he, scale=10, state_key="sequence")
za.calibrate(recs, min_rows=4, act_accuracy=0.9)
aa = za.predict(recs[0]["state"], qd)["dept"]
test_eq((sorted(aa["action"]), aa["action"]["act_probability"]), (["act", "act_probability"], aa["answer_confidence"]))
test_eq("action" in zk.predict(recs[0]["state"], qd)["dept"], False)            # calibrated without act_accuracy: no action
aa["action"]

{'act_probability': 0.9999, 'act': True}

In [ ]:
m = evaluate(zk, recs)
test_eq(m["accuracy"], 1.0)
{k: round(m[k], 3) for k in ("accuracy", "brier", "ece", "macro_f1")} | {"temperatures": temps}

{'accuracy': 1.0,
 'brier': 0.001,
 'ece': 0.008,
 'macro_f1': 1.0,
 'temperatures': {'choice': 0.5, 'choice:2': 0.5}}

The state tower is usually the expensive one (a protein language model, an image tower), and the option texts are few. So a decider embeds each distinct state once per call, however many questions ask about it, and remembers each option text. Comparing prompts means one decider per prompt over the same states. Wrapping the state tower in `CachedEmbedder` lets them share its vectors, so each state is embedded once for all of them:

In [ ]:
class CountingEmbedder(HashEmbedder):
    "A HashEmbedder that counts what it embeds"
    n = 0
    def embed(self, items): self.n += len(items); return super().embed(items)

ce = CountingEmbedder()
SimilarityDecider(ce, he, scale=10, state_key="sequence").calibrate(recs, min_rows=4)
test_eq(ce.n, 4)                                      # 16 records, 4 distinct sequences
shared = CachedEmbedder(ce := CountingEmbedder())
for prompt in ("{text}", "a ticket about {text}", "{key}"):
    SimilarityDecider(shared, he, scale=10, prompt=prompt, state_key="sequence").predict_batch([r["state"] for r in recs], qd)
test_eq(ce.n, 4)                                      # three prompts, each sequence embedded once

## A masked language model: verbalizers

A masked language model already predicts words: put the options in the prompt with a label each, end the prompt with a mask, and read what the model would write there. Each option is represented by one *verbalizer* token, its letter (A, B, C…) for a choice, its digit for a score level, *yes* or *no* for a noul, and its logit is the model's logit for that token at the mask. The softmax over the verbalizers is the answer, and a temperature fitted on held-out records calibrates it. This is prompt-based classification (PET), and the zero-shot path the protein note calls Path 1. Which letters or digits stand for the options depends on the kind, through `verbalizer_options`, a generic function like `option_texts`.

It works to the extent that the model saw prompts of this shape in pre-training, and its answers move with the prompt's wording: try a few phrasings and report the best and the average. Letters are used because an option's own words often break into several tokens, while a mask predicts one.

In [ ]:
#| export
VERBALIZER_PROMPT = "{state}\nQuestion: {instructions}\n{options}\nAnswer: {mask}"

@dispatch
def verbalizer_options(q:Choice) -> tuple:
    "A choice's verbalizers and option texts: letters, and `label: description`"
    return [chr(65 + i) for i in range(q.k)], [str(k) if v in (None, "") else f"{k}: {render_criterion(v)}" for k, v in q.criteria.items()]

@dispatch
def verbalizer_options(q:Score) -> tuple:
    "A score's verbalizers and option texts: its levels' digits, and their descriptions"
    return [str(i) for i in range(q.k)], [render_criterion(c) for c in q.criteria]

@dispatch
def verbalizer_options(q:Noul) -> tuple:
    "A noul's two sides as lettered options, for a tokenizer without one-token yes and no"
    return ["A", "B"], q.options

def _one_token(tok, words) -> int|None:
    "The id of the first of `words` the tokenizer keeps as one token, with a leading space or without"
    for w in words:
        for s in (" " + w, w):
            ids = tok(s, add_special_tokens=False)["input_ids"]
            if len(ids) == 1: return ids[0]
    return None

class VerbalizerDecider:
    "Zero-shot typed answers from a masked language model's head: each option's logit is the model's logit for its verbalizer token at the prompt's mask"
    def __init__(self,
                 model, # a transformers `...ForMaskedLM`
                 tokenizer, # its tokenizer, with a mask token
                 prompt:str=VERBALIZER_PROMPT, # {state}, {instructions}, {options} and {mask}
                 temperatures:dict|None=None, act_threshold:float|dict|None=None, digits:int|None=4, max_len:int=512, batch_size:int=8,
                 device=None, name:str="verbalizer"):
        store_attr(but="model,temperatures,device")
        self.model, self.temperatures = model.eval(), dict(temperatures or {})
        self.device = torch.device(device) if device is not None else next(model.parameters()).device
        self.model.to(self.device)
        full = tokenizer("x", add_special_tokens=True)["input_ids"]; x = tokenizer("x", add_special_tokens=False)["input_ids"]
        at = next(i for i in range(len(full)) if full[i:i + len(x)] == x)
        self._head, self._tail = full[:at], full[at + len(x):]           # the tokenizer's own special tokens around a text

    @classmethod
    def from_pretrained(cls, id:str, **kw):
        "The decider over a Hub (or local) masked language model"
        from transformers import AutoModelForMaskedLM, AutoTokenizer
        return cls(AutoModelForMaskedLM.from_pretrained(id), AutoTokenizer.from_pretrained(id), **kw)

    def __repr__(self): return f"VerbalizerDecider({self.name} · {type(self.model).__name__} · {len(self.temperatures)} temperatures)"

    def verbalizers(self, q:Question) -> tuple:
        "(the options' text in the prompt, each option's verbalizer token id)"
        if isinstance(q, Noul):
            ids = [_one_token(self.tokenizer, ["no", "No", "false", "False"]), _one_token(self.tokenizer, ["yes", "Yes", "true", "True"])]
            if all(i is not None for i in ids): return "Answer yes or no.", ids
        marks, descr = verbalizer_options(q)            # a noul without one-token yes and no: its sides as lettered options
        ids = [_one_token(self.tokenizer, [m, m.lower()]) for m in marks]
        if any(i is None for i in ids): raise ValueError(f"question {q.name!r}: a verbalizer is not one token of this tokenizer")
        return "Options: " + "; ".join(f"{m}. {d}" for m, d in zip(marks, descr)), ids

    def _ids(self, state, q:Question) -> tuple:
        "The prompt's token ids, the mask's position and the verbalizer ids; the state is cut to fit `max_len`"
        text, vids = self.verbalizers(q)
        before, after = self.prompt.split("{state}")
        tail = after.format(instructions=q.instructions, options=text, mask=self.tokenizer.mask_token)
        st = self.tokenizer(before + render_state(_parsed(state)), add_special_tokens=False)["input_ids"]
        tl = self.tokenizer(tail, add_special_tokens=False)["input_ids"]
        room = max(0, self.max_len - len(tl) - len(self._head) - len(self._tail))
        ids = self._head + st[:room] + tl + self._tail
        return ids, len(ids) - 1 - ids[::-1].index(self.tokenizer.mask_token_id), vids

    @torch.no_grad()
    def row_logits(self, records:list, questions:list) -> list:
        "Each (record, question) pair's option logits, the model's logits for the verbalizers at the mask"
        built = [self._ids(r["state"], q) for r, q in zip(records, questions)]
        out, pad = [], self.tokenizer.pad_token_id or 0
        for i in range(0, len(built), self.batch_size):
            chunk = built[i:i + self.batch_size]; L = max(len(b[0]) for b in chunk)
            ids = torch.full((len(chunk), L), pad, dtype=torch.long); att = torch.zeros_like(ids)
            for j, (x, _, _) in enumerate(chunk): ids[j, :len(x)] = torch.tensor(x); att[j, :len(x)] = 1
            logits = self.model(input_ids=ids.to(self.device), attention_mask=att.to(self.device)).logits.float()
            for j, (_, m, v) in enumerate(chunk): out.append(logits[j, m, v].cpu().numpy())
        return out

    def _answer(self, q:Question, z) -> Answer:
        z = np.asarray(z, dtype=np.float64) / lookup_temperature(self.temperatures, q.qtype, q.k)
        p = np.exp(z - z.max()); p /= p.sum()
        return fill_answer(q, p, self.digits, act=float(p.max()) if self.act_threshold is not None else None, act_threshold=self.act_threshold)

    def predict_batch(self, states:list, questions:dict) -> list:
        "Answers for many states, one prompt per state and question"
        qs = as_questions(questions)
        pairs = [(s, q) for s in states for q in qs.values()]
        z = self.row_logits([{"state": s} for s, _ in pairs], [q for _, q in pairs])
        k = len(qs)
        return [{q.name: self._answer(q, z[i * k + j]) for j, q in enumerate(qs.values())} for i in range(len(states))]

    def predict(self, state, questions:dict) -> dict: return self.predict_batch([state], questions)[0]

    def calibrate(self, records:list, min_rows:int=10, act_accuracy:float|None=None) -> dict:
        "Fit temperatures per question type and option-count bucket on labelled records; `act_accuracy` also fits act thresholds"
        self.temperatures, thr = _fit_zero_shot(self, records, min_rows, act_accuracy)
        if thr is not None: self.act_threshold = thr
        return self.temperatures

On the fixture's tiny ModernBERT, whose masked-LM head is untrained, the answers are noise, but the prompt, the verbalizers and the schema are the real ones:

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

In [ ]:
FIX = "fixtures/tiny-modernbert"
vd = VerbalizerDecider(AutoModelForMaskedLM.from_pretrained(FIX), AutoTokenizer.from_pretrained(FIX), device="cpu")
qv = {"dept": {"type": "choice", "instructions": "Which team?", "criteria": {"billing": "invoices", "tech": "bugs", "sales": "pricing"}},
      "urgent": {"type": "noul", "instructions": "Is it urgent?"},
      "level": {"type": "score", "instructions": "How angry?", "criteria": ["calm", "annoyed", "furious"]}}
ids, m, v = vd._ids("The app crashes when I pay.", Question.from_dict(qv["dept"], name="dept"))
test_eq(ids[m], vd.tokenizer.mask_token_id)
print(vd.tokenizer.decode(ids))
va = vd.predict("The app crashes when I pay.", qv)
test_close(sum(va["dept"]["probabilities"].values()), 1.0, eps=1e-3)
test_eq((set(va["dept"]["probabilities"]), va["urgent"]["type"], va["level"]["type"]), ({"billing", "tech", "sales"}, "noul", "score"))
long = vd._ids("word " * 2000, Question.from_dict(qv["dept"], name="dept"))[0]
test_eq(len(long) <= vd.max_len, True)                                        # a long state is cut, the question and the mask kept
va

[CLS]The app crashes when I pay.
Question: Which team?
Options: A. billing: invoices; B. tech: bugs; C. sales: pricing
Answer:[MASK][SEP]


{'dept': {'type': 'choice',
  'choice': 'tech',
  'probabilities': {'billing': 0.3078, 'tech': 0.3612, 'sales': 0.331},
  'confidence': 0.002,
  'answer_confidence': 0.3612},
 'urgent': {'type': 'noul',
  'noul': 0.5005,
  'confidence': 0.5005,
  'answer_confidence': 0.5005},
 'level': {'type': 'score',
  'score': 1.0302,
  'legend': {'0': 'calm', '1': 'annoyed', '2': 'furious'},
  'probabilities': {'0': 0.3372, '1': 0.2954, '2': 0.3674},
  'confidence': 0.0036,
  'answer_confidence': 0.3674}}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()